In [1]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
plt.rcParams['font.family'] = 'Times New Roman'
#plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['mathtext.default'] = 'regular'
from scipy.optimize import curve_fit, minimize
from scipy.stats import norm, poisson
from scipy.special import gamma
from pathlib import Path
# import natural units:
import natural_units as nu
# multi-core/thread:
import concurrent.futures
import math
import matplotlib.ticker as ticker
from scipy.integrate import quad
def gaussian(x, mean, stddev):
    return  np.exp(-((x - mean) ** 2) / (2 * stddev ** 2)) / stddev / np.sqrt(2*np.pi)
def poisson(x, lambda_param):
    return np.exp(-lambda_param) * np.power(lambda_param, x) / gamma(x+1) #if not lambda_param < 1e-9 else
def log_poisson(x, lambda_param):
    if x > 0:
        return x * np.log(lambda_param) - lambda_param - (x*np.log(x) - x + np.log(2*np.pi*x)/2 + 1/12/x)  # 这 lambda_param 不能是0。如果很小，结果很负，摆动很大，注意。log(N!) = N*log(N) - N + log(2pi N)/2 + 1/12/N - ...
    else:
        return -lambda_param

In [2]:
input_file_dir = Path('./results/masked')
input_files = [file for file in input_file_dir.iterdir() if file.suffix == ".txt"]

In [3]:
dn_min = -200
dn_max = 400
dn_range = range(dn_min, dn_max)
log_m_min  = -3
log_m_max  = 1
n_m    = 17   # From 1e-3  to 10 GeV
#log cs shift from the balloon line
cs_logshift_min = -3
cs_logshift_max = 0
cs_shift_numbers   = 25
m_grid      = np.logspace(log_m_min, log_m_max, n_m)
cs_logshift = np.linspace(cs_logshift_min, cs_logshift_max, cs_shift_numbers)
center_line = np.array([2.15504637e-23, 1.53030461e-23, 1.26359147e-23, 1.61669130e-23,
       2.40008514e-23, 4.03532201e-23, 6.95747264e-23, 1.40957345e-22,
       2.84518575e-22, 5.17381239e-22, 1.08351297e-21, 2.15203017e-21,
       4.12016417e-21, 7.78952222e-21, 1.45615810e-20, 2.70914228e-20,
       4.94000000e-20])
signal_gird = np.loadtxt('../SHIELDING_RESULT/signal_grid.txt')

def process_file(filename):
    input_file_base = filename.stem
    N_dn     = np.loadtxt(filename)[:,1]
    peak = np.argmax(N_dn) + dn_min
    dn_min_likelihood = peak - 50
    dn_max_likelihood = peak + 50 + 1 #难绷
    total_count = np.sum(N_dn)
    log_bottom = 0
    for a in N_dn:
        if a > 20:
            log_bottom += a * np.log(a) - a
        else:
            log_bottom += np.log(math.factorial(int(a)))

    # x[0]:peak_rescale  x[1]:peak_center  x[2]:read_noise  x[3]: second gaussian portion  x[4]: second gaussian
    def BKD_double_G(x):
        bkd_analytical = np.zeros(dn_max-dn_min)
        for dn in range(0, 100):
            poisson_dn = poisson(dn, x[1])
            smear_min = dn_min
            smear_max = dn_max
            for dn_smear in range(smear_min, smear_max):
                bkd_analytical[dn_smear-dn_min] += poisson_dn * ((1-x[3])*gaussian(dn_smear, dn, x[2])+x[3]*gaussian(dn_smear, dn, x[4]))
        bkd_analytical = bkd_analytical * total_count * x[0]
        log_likelihood_0 = 0
        for k in range(dn_min_likelihood-dn_min,dn_max_likelihood-dn_min):
            log_likelihood_0 += log_poisson(N_dn[k], bkd_analytical[k])
        return log_likelihood_0-log_bottom, bkd_analytical
    def log_likelihood_bkd_inverse_double_G(x):
        return -BKD_double_G(x)[0]
    initial_guess_bkd = [1, peak, 13, 0.1, 18]
    bounds = [(0.9,1.1),(0,np.max((1.5*peak, 20))),(8,16),(0,1),(16,30)]
    fitting_result_bkd = minimize(log_likelihood_bkd_inverse_double_G, initial_guess_bkd, bounds=bounds, method= 'SLSQP')
    bkd  = BKD_double_G(fitting_result_bkd.x)[1]
    fig, ax = plt.subplots(figsize = (4,3), dpi = 200)
    # 然后 chi_2 是个什么东西？  lambda=L(F_obs,F_model)/L(F_model,F_model)
    # -2 log lambda
    log_lambda_fitting = 0
    chi_2 = 0
    for k in range(dn_min_likelihood-dn_min,dn_max_likelihood-dn_min):
        log_lambda_fitting += log_poisson(N_dn[k], bkd[k]) - log_poisson(bkd[k], bkd[k])
        chi_2 += (N_dn[k]-bkd[k])**2 / bkd[k]
    #print(-2 * log_lambda_fitting)
    #print(chi_2)
    fontsize = 9
    plt.semilogy(dn_range, N_dn, color = 'blue', linewidth=0.4)
    plt.semilogy(dn_range, bkd, color = 'green', linewidth=0.4)
    plt.axvline(x=dn_min_likelihood, color='black', linestyle='--', linewidth = 0.3)
    plt.axvline(x=dn_max_likelihood, color='black', linestyle='--', linewidth = 0.3)
    ax.set_title('Pixel Charge Distribution\n obs_id:  ' + input_file_base, fontsize=fontsize)
    ax.set_xlabel(r'$N_e$ (number of charges)', fontsize=fontsize)
    ax.set_ylabel(r'$F_{obs}(N_e)$ (number of pixels)', fontsize=fontsize)
    ax.set_xlim(-80,100)
    ax.set_ylim(1,1e5)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.tick_params(axis="both", direction="in", which = 'major', top=False, right=False, width = 0.2)
    ax.tick_params(axis="both", direction="in", which = 'minor', top=False, right=False, width = 0.2)
    ax.tick_params(labeltop=False, labelright=False)
    ax.text(-30, 2, 'h = {}\nDC = {}\nsigma1 = {}\nmixing = {}\nsigma2 = {}\nchi2 = {}'.format(fitting_result_bkd.x[0],fitting_result_bkd.x[1],fitting_result_bkd.x[2],fitting_result_bkd.x[3],fitting_result_bkd.x[4],-2 * log_lambda_fitting), fontsize=fontsize-2, color='black', bbox=dict(facecolor='white', edgecolor='black', boxstyle='square',linewidth = 0.5))
    for spine in ax.spines.values():
        spine.set_linewidth(0.2)  # 设置边框线宽为 2
    legend_elements = [
        Line2D([0], [0], color='green', linestyle='-', linewidth = 0.5, label='Background-only fitting')
    ]
    plt.legend(handles=legend_elements, fontsize=7, loc='upper right', handletextpad=0.4, frameon=True)
    plt.savefig('./results/masked/'+input_file_base+'_bkd_fittings.pdf', format="pdf",bbox_inches="tight")
    plt.show()

In [4]:
def process_files_in_parallel(file_list):
    with concurrent.futures.ProcessPoolExecutor() as executor:
        executor.map(process_file, file_list)
process_files_in_parallel(input_files)